In [2]:
from dataclasses import dataclass
import math
import torch
import torch.nn as nn
from torch.nn import functional as F


# ------------------------------------------------------------------------
@dataclass
class GPTConfig:
    block_size: int = 1024 #T
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768 #C


class CausalSelfAttention(nn.Module):
    
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0        
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd) # key, query, Value projections for all heads        
        self.c_proj = nn.Linear(config.n_embd, config.n_embd) # projection
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer('bias', torch.tril(torch.ones(config.block_size, config.block_size))
                             .view(1, 1, config.block_size, config.block_size)) # (T, T) ->  (1,1,T,T)
        
    def forward(self, x):
        B, T, C = x.size() # (B, T, n_embd)
        
        qkv = self.c_attn(x) # (B, T, 2304(3 * n_embd))
        q, k, v = qkv.split(self.n_embd, dim=2) #3 * (B, T, 768(n_embd))) split in dim2, which is C dim         
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) #(B, nh, T, hs) (B, 12, T, 64)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) #(B, nh, T, hs) (B, 12, T, 64)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) #(B, nh, T, hs) (B, 12, T, 64)

        # (B,nh,T,hs) @ (B,nh,hs,T) -> (B,nh,T,T), then sqrt(hs)
        att = q @ k.transpose(-2, -1) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1) # (B,nh,T,T)
        y = att @ v # (B,nh,T,T) @ (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y) # output projection
        return y


class MLP(nn.Module):
    
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        
    def forward(self, x):
        x = self.c_fc(x) # (B,T, n_embd) ->  (B,T, 4 * n_embd) 
        x = self.gelu(x) 
        x = self.c_proj(x) # (B,T, 4 * n_embd) ->  (B,T, n_embd) 
        return x 
        

class Block(nn.Module):
    
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)
        
    def forward(self, x):
        x = x + self.attn(self.ln_1(x)) # (B,T, n_embd) -> (B,T, n_embd)
        x = x + self.mlp(self.ln_2(x))  # (B,T, n_embd) -> (B,T, n_embd)
        return x 

    
class GPT(nn.Module):
    
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd), # weight token embedding 
            wpe  = nn.Embedding(config.block_size, config.n_embd), # position token embedding 
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]), #
            ln_f = nn.LayerNorm(config.n_embd), #layernorm final
        ))        
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
    
    def forward(self, idx):
        #idx is B, T
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is {self.config.block_size}"
        # embedding
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device) #shape(T)  [0,1,2,...,T-1]
        pos_emb = self.transformer.wpe(pos) # (T, n_embd)
        tok_emb = self.transformer.wte(idx) # (B, T, n_embd)
        x = tok_emb + pos_emb # (B, T, n_embd)
        
        # transformer
        for block in self.transformer.h: # self attention and MLP
            x = block(x) # (B, T, n_embd)
        
        # final layer norm and the classifier
        x = self.transformer.ln_f(x) # (B, T, n_embd)
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        return logits
                
    @classmethod
    def from_pretrained(cls, model_type):
        """ load pretrained GPT2 model weights from huggingface """
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # 1) Based on model_type
        config_args = {
            'gpt2':        dict(n_layer=12, n_head=12, n_embd=768),   # 124M
            'gpt2-medium': dict(n_layer=24, n_head=16, n_embd=1024),  # 350M
            'gpt2-large':  dict(n_layer=36, n_head=20, n_embd=1280),  # 774M
            'gpt2-xl':     dict(n_layer=48, n_head=25, n_embd=1600),  # 1558M
        }[model_type]
        config_args['vocab_size'] = 50257   # alwas 50257 for GPT-2 
        config_args['block_size'] = 1024    # alwas 1024 for GPT-2 

        # 2 create a initialized gpt model
        config = GPTConfig(**config_args)
        model = cls(config)
        sd = model.state_dict()
        sd_keys = sd.keys()        
        # attn.bias(down tril mask) is buffer, no need to learn
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')]

        # 3 HF model，
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()
        sd_keys_hf = sd_hf.keys()
        
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')]
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')]

        # These are need .t()
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight',
                    'mlp.c_fc.weight', 'mlp.c_proj.weight']

        assert len(sd_keys_hf) == len(sd_keys), \
            f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"

        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                assert sd_hf[k].shape[::-1] == sd[k].shape   
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model
    


In [3]:
model = GPT.from_pretrained('gpt2')
model.eval()

loading weights from pretrained gpt: gpt2


GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode("Hello, I'm a language model,")
tokens = torch.tensor(tokens, dtype=torch.long)
tokens

tensor([15496,    11,   314,  1101,   257,  3303,  2746,    11])

In [5]:
num_return_seq = 5
max_length = 30

tokens = tokens.unsqueeze(0).repeat(num_return_seq, 1)
tokens

tensor([[15496,    11,   314,  1101,   257,  3303,  2746,    11],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11]])

In [7]:
x = tokens
x.dtype, x.device, x.shape

(torch.int64, device(type='cpu'), torch.Size([5, 8]))

In [ ]:
# x (B, T) 
torch.manual_seed(42)

while x.size(1) < max_length:
    with torch.no_grad():
        logits = model(x) #(B, T, vocab_size)
        logits = logits [:, -1, :] #(B, vocab_size)
        probs = F.softmax(logits, dim=-1) # (B, vocab_size)
        
        #top-K
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)  # (B, 50) from big to small,  (B, 50)
        ix = torch.multinomial(topk_probs, 1) # (B, 1), ix is the index in topk_probs e.g. ix   # (B, 1)  比如 [[0], [3], [1], [7], [0]]
        xcol = torch.gather(topk_indices, -1, ix) # get the real idx, use ix as index, search form topk_indices
        x = torch.cat((x, xcol), dim=-1)
x

tensor([[15496,    11,   314,  1101,   257,  3303,  2746,    11,   407,   257,
          8300,  3859,     0,   314,   655,   787,  5370,  1912,   319,   584,
          4493,    13,   314,  1949,   284,   466,   326,   526,   198,   198],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11,   257,  1611,
           286,   257,   366, 11085,  1398,  9511,     1,   286,   262,   995,
           290,   257,  1048,   326,  2058,   422,   257,   881,   517, 41469],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11,   290,   314,
          1101,  3599,   284,  1561,   546,   262,  9495,   286,   262, 15582,
            11,   290,   314,  1101,   635,  1762,   319,   281,  7552,   326],
        [15496,    11,   314,  1101,   257,  3303,  2746,    11,   780,   314,
          1101,  3597,  1103,    12,  2435,    13,   314,  1101,  3597,   477,
          8950,    13,   843,   314,  1101,  1762,   351,  8950,   329,   502],
        [15496,    11,   314,  1101,   257,  330

In [12]:
for i in range(num_return_seq):
    tokens = x[i, :max_length].tolist()
    decoded = enc.decode(tokens)
    print(">", decoded)

> Hello, I'm a language model, not a programming platform! I just make decisions based on other projects. I try to do that."


> Hello, I'm a language model, a kind of a "first class citizen" of the world and a person that comes from a much more egalitarian
> Hello, I'm a language model, and I'm starting to talk about the notion of the syntax, and I'm also working on an extension that
> Hello, I'm a language model, because I'm writing real-time. I'm writing all languages. And I'm working with languages for me
> Hello, I'm a language model, I don't know where to begin but I know there is a big deal going on with our society. What
